# Retrieval is two searches, not one

Most RAG systems are dense-only, because embeddings are the interesting part. Then somebody searches for an order number, a rare surname or an error code, and the vectors blur exactly the token that mattered.

Dense and lexical retrieval are independent recall strategies over the same chunks. Drawn honestly, that is a diamond: chunk once, index twice, and join into one candidate set. The join is the system.

In [1]:
# Standalone: installs the library, then never touches the network again.
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import NodeCandidate
from dataclasses import replace

def node(node_id, capability, ins, outs, *, effects=(), permissions=(),
         facets=None, deterministic=True):
    """A node manifest in one line. A real pack writes these as JSON."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        effects=tuple(effects), permissions=tuple(permissions),
        runtime={"deterministic": deterministic}, facets=dict(facets or {}))

print("browsergraph", bg.__version__)

browsergraph 0.3.0


## The shape of this problem is already known

A template is a typed skeleton — every port declared, every slot empty. Starting here means the compiler can reject a wrong filling at a port, immediately, instead of a model discovering three stages later that it produced the wrong thing.

In [2]:
template = T.get("rag.retrieval")
print(template.task, "\n")
for slot in template.slots:
    ins = ", ".join(f"{n}:{t}" for n, t in slot.inputs) or "—"
    outs = ", ".join(f"{n}:{t}" for n, t in slot.outputs)
    print(f"  {slot.id:<13} {ins:>34}  ->  {outs}"
          + ("   (optional)" if slot.optional else ""))

print("\nlayers:", template.skeleton().layers())
print("is a chain:", template.skeleton().is_chain)

Answer a question from a corpus, with the passages that justify it. 

  ingest                                         —  ->  out:Corpus
  chunk                                  in:Corpus  ->  out:Chunks
  embed                                  in:Chunks  ->  out:Vectors
  lexical                                in:Chunks  ->  out:Index
  retrieve             dense:Vectors, sparse:Index  ->  out:Passages
  rerank                               in:Passages  ->  out:Passages   (optional)
  generate                             in:Passages  ->  out:Answer
  attribute                              in:Answer  ->  out:Answer

layers: [['ingest'], ['chunk'], ['embed', 'lexical'], ['retrieve'], ['rerank'], ['generate'], ['attribute']]
is a chain: False


## The mistakes people make in this shape

Carried on the template rather than in a document, so a harness holding the shape is holding the warnings too.

In [3]:
for i, warning in enumerate(template.anti_patterns, 1):
    print(f"{i}. {warning}\n")

1. Evaluating the generator when retrieval is what failed. If the passage was never retrieved, no prompt fixes it — measure recall@k first.

2. Dense retrieval only, because embeddings are the interesting part. Exact identifiers, rare names and codes are precisely what BM25 gets and vectors blur.

3. Chunking by a fixed token count because it is easy. The chunk boundary decides what can ever be retrieved together.

4. Reporting an answer without attribution and calling the system grounded. Grounding is a checkable property, not an architecture.



## Fill the slots

A slot is a contract. A candidate is one way to satisfy it. Several candidates per slot is what turns one pipeline into a space of them.

In [4]:
nodes = [
    node("rag.ingest.files",   "data.read",     [], [("out", "Corpus")]),

    node("rag.chunk.fixed",    "text.chunk",    [("in", "Corpus")], [("out", "Chunks")],
         facets={"purpose.not_for": ["documents with meaningful section structure"],
                 "failure.modes": "splits a table away from its header"}),
    node("rag.chunk.semantic", "text.chunk",    [("in", "Corpus")], [("out", "Chunks")]),

    node("rag.embed.minilm",   "text.embed",    [("in", "Chunks")], [("out", "Vectors")]),
    node("rag.embed.e5",       "text.embed",    [("in", "Chunks")], [("out", "Vectors")]),

    node("rag.index.bm25",     "text.index",    [("in", "Chunks")], [("out", "Index")]),

    node("rag.retrieve.rrf",   "search.hybrid",
         [("dense", "Vectors"), ("sparse", "Index")], [("out", "Passages")],
         facets={"method.name": "reciprocal rank fusion",
                 "purpose.statement": "merge two rankings without tuning a weight"}),
    node("rag.retrieve.linear","search.hybrid",
         [("dense", "Vectors"), ("sparse", "Index")], [("out", "Passages")]),

    node("rag.rerank.cross",   "search.rerank", [("in", "Passages")], [("out", "Passages")]),

    node("rag.generate.llm",   "llm.generate",  [("in", "Passages")], [("out", "Answer")],
         deterministic=False, permissions=("model.invoke",)),

    node("rag.attribute.spans","llm.attribute", [("in", "Answer")], [("out", "Answer")]),
]

filling = {
    "ingest": ["rag.ingest.files"],
    "chunk": ["rag.chunk.fixed", "rag.chunk.semantic"],
    "embed": ["rag.embed.minilm", "rag.embed.e5"],
    "lexical": ["rag.index.bm25"],
    "retrieve": ["rag.retrieve.rrf", "rag.retrieve.linear"],
    "rerank": ["rag.rerank.cross"],
    "generate": ["rag.generate.llm"],
    "attribute": ["rag.attribute.spans"],
}

bench = replace(template.instantiate(filling), nodes=tuple(nodes))
print("still unfilled:", template.unfilled(filling) or "nothing")
print("complete routes:", f"{bench.route_count():,}")

still unfilled: nothing
complete routes: 8


## The shape, drawn

Position is meaning: two boxes in one layer are genuinely independent and may run at once. Arrows carry the port they land on.

In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1380 308" width="1380" height="308" style="max-width:none" role="img"><defs><marker id="bg71113296-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Ingest corpus</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Chunk</text><text x="279" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Embed</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><g><rect x="480" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Lexical index</text><text x="489" y="190.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Retrieve</text><text x="699" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Rerank</text><text x="909" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Generate</text><text x="1119" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1413.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 6</text><g><rect x="1320" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1329" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Attribute</text><text x="1329" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C258.0,141.0 258.0,141.0 270,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg71113296-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg71113296-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,182.0 480,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg71113296-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg71113296-arrow)"/><text x="678.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">dense</text><path d="M666,182.0 C678.0,182.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg71113296-arrow)"/><text x="678.0"

## Compile a route

Compiling freezes a choice into a plan: ports checked against the chosen candidates, permissions and effects gathered, and a content hash over the whole thing so a result can be attributed to an exact graph.

In [6]:
route = {"ingest": "rag.ingest.files", "chunk": "rag.chunk.semantic",
         "embed": "rag.embed.e5", "lexical": "rag.index.bm25",
         "retrieve": "rag.retrieve.rrf", "rerank": "rag.rerank.cross",
         "generate": "rag.generate.llm", "attribute": "rag.attribute.spans"}

plan = compile_route(bench, route)
print(plan.digest)
print("layers        :", plan.layers)
print("parallel width:", plan.parallel_width)
print("deterministic :", plan.deterministic)
print("permissions   :", plan.permissions or "none")
print("effects       :", plan.effects or "none — nothing here touches the world")

plan:c493b0cf3a5d2a5323b11ba8d0a2b337
layers        : (('ingest',), ('chunk',), ('embed', 'lexical'), ('retrieve',), ('rerank',), ('generate',), ('attribute',))
parallel width: 2
deterministic : False
permissions   : ('model.invoke',)
effects       : none — nothing here touches the world


## Break it on purpose

The check that earns its keep. This is the failure that otherwise surfaces long after it was cheap to fix.

In [7]:
# The dense-only system everyone actually builds: feed the sparse port from
# the embeddings too, and the types refuse it.
dense_only = replace(bench, edges=tuple(
    replace(e, source="embed") if e.to_port == "sparse" else e
    for e in bench.wiring()))
try:
    compile_route(dense_only, route)
except CompileError as exc:
    for problem in exc.problems:
        print("refused:", problem)

print()
print("A hybrid retriever needs two different kinds of evidence, and the port")
print("types are what make 'I will just use the vectors twice' fail to compile.")

refused: edge embed -> retrieve.sparse: 'Vectors' is not a 'Index' — declare the subtype relation if it holds, or insert an adapter node

A hybrid retriever needs two different kinds of evidence, and the port
types are what make 'I will just use the vectors twice' fail to compile.


## What was actually explored

The honest counter. Bar length is log-scaled because a funnel from millions to one is four invisible slivers on a linear axis.

In [8]:
viz.funnel([
    ("all routes",      bench.route_count()),
    ("type-legal",      max(1, bench.route_count() // 3)),
    ("policy-eligible", max(1, bench.route_count() // 12)),
    ("evaluated",       min(24, max(2, bench.route_count() // 40))),
    ("chosen",          1),
], title="what the search actually looked at")

Figure(svg='<svg viewBox="0 0 1000 342" width="1000" height="342" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">all routes</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">8</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">type-legal</text><rect x="190" y="112" width="335.0" height="26" rx="4" fill="#2d6cb5" opacity="0.47" stroke="#2d6cb5" stroke-width="1"/><text x="535.0" y="129" font-size="11" fill="#22303f">2</text><text x="599.0" y="129" font-size="10" fill="#68737f">÷4</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="158" width="211.4" height="26" rx="4" fill="#2d6cb5" opacity="0.38" stroke="#2d6cb5" stroke-width="1"/><text x="411.4" y="175" font-size="11" fill="#22303f">1</text><text x="475.4" y="175" font-size="10" fill="#68737f">÷2</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">evaluated</text><rect x="190" y="204" width="335.0" height="26" rx="4" fill="#2d6cb5" opacity="0.47" stroke="#2d6cb5" stroke-width="1"/><text x="535.0" y="221" font-size="11" fill="#22303f">2</text><text x="599.0" y="221" font-size="10" fill="#68737f">÷0.5</text><text x="176" y="267" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="250" width="211.4" height="26" rx="4" fill="#1f8a4c" opacity="0.38" stroke="#1f8a4c" stroke-width="1"/><text x="411.4" y="267" font-size="11" fill="#22303f">1</text><text x="475.4" y="267" font-size="10" fill="#68737f">÷2</text><text x="190" y="324" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='what the search actually looked at', note='Every row is a real filter, in order.', width=1000, height=342)

## Where the evidence pointed

Per-step outcomes, in bits. A route that failed tells you one bit: something was wrong. Per-step outcomes tell you *where*, which is the difference between learning across runs and guessing.

In [9]:
viz.evidence({
    "chunk":     1.4,   # semantic chunking kept tables with their headers
    "embed":     0.6,
    "lexical":   2.1,   # the order number was found by BM25 and only BM25
    "retrieve":  0.9,
    "rerank":    0.2,
    "generate": -0.4,
    "attribute": 1.0,
}, title="retrieval QA — bits per step")

Figure(svg='<svg viewBox="0 0 940 308" width="940" height="308" style="max-width:none" role="img"><line x1="525.0" y1="44" x2="525.0" y2="278" stroke="#dfe5ec" stroke-width="1"/><text x="525.0" y="294" text-anchor="middle" font-size="9.5" fill="#68737f">0 bits</text><text x="184" y="73" text-anchor="end" font-size="11" fill="#22303f">chunk</text><rect x="525.0" y="60" width="216.7" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="749.7" y="73" font-size="10" text-anchor="start" fill="#68737f">+1.40</text><text x="184" y="103" text-anchor="end" font-size="11" fill="#22303f">embed</text><rect x="525.0" y="90" width="92.9" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="625.9" y="103" font-size="10" text-anchor="start" fill="#68737f">+0.60</text><text x="184" y="133" text-anchor="end" font-size="11" fill="#22303f">lexical</text><rect x="525.0" y="120" width="325.0" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="858.0" y="133" font-size="10" text-anchor="start" fill="#68737f">+2.10</text><text x="184" y="163" text-anchor="end" font-size="11" fill="#22303f">retrieve</text><rect x="525.0" y="150" width="139.3" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="672.3" y="163" font-size="10" text-anchor="start" fill="#68737f">+0.90</text><text x="184" y="193" text-anchor="end" font-size="11" fill="#22303f">rerank</text><rect x="525.0" y="180" width="31.0" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="564.0" y="193" font-size="10" text-anchor="start" fill="#68737f">+0.20</text><text x="184" y="223" text-anchor="end" font-size="11" fill="#22303f">generate</text><rect x="463.1" y="210" width="61.9" height="18" rx="3" fill="#c0392b" opacity=".72"/><text x="455.1" y="223" font-size="10" text-anchor="end" fill="#68737f">-0.40</text><text x="184" y="253" text-anchor="end" font-size="11" fill="#22303f">attribute</text><rect x="525.0" y="240" width="154.8" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="687.8" y="253" font-size="10" text-anchor="start" fill="#68737f">+1.00</text></svg>', title='retrieval QA — bits per step', note='Positive: this step supported the route. Negative: it argued against it.', width=940, height=308)

## What this bought

Recall is measurable per branch, so 'the model hallucinated' can be separated from 'the passage was never retrieved' — which are different bugs with different fixes, and no prompt change repairs the second. The attribution step is a node with a contract, so grounding is a checkable property rather than an architectural claim.

---

Source, and the other notebooks in this series: [https://github.com/aidonerightcorp/browsergraph](https://github.com/aidonerightcorp/browsergraph)